# 3. Prediction — version 2
Reads the fit written by `modelling_code_version2.ipynb` (via `active_fit.json` and `bundle.json`), so
no model name or folder needs to be typed. Self-contained: all imports are in the first cell.

Kept from your version-0 prediction code: `PRED_MEAN` + `PRED_Q45/55/70/75`, forecast in blocks, one
combined parquet, series/s and ETA printout, negative-binomial decoding with the built-in self-test.

Changed: model, series, statics and covariates come from the fit folder; inference runs in float32
(bf16 can push the distribution parameters out of range); artifact hashes are verified before use;
the output is written inside the fit folder.

In [ ]:
from pathlib import Path
import os, json, hashlib, gc, pickle
from collections.abc import Sequence
from datetime import datetime
from uuid import uuid4
import numpy as np
import pandas as pd
import torch, darts
import pytorch_lightning as pl
from scipy import stats
from darts import TimeSeries
from darts.models import TFTModel
from darts.utils.likelihood_models import NegativeBinomialLikelihood


def digest(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(8 * 1024 * 1024), b""):
            h.update(b)
    return h.hexdigest()


def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


def write_json(path, obj):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(obj, indent=2, allow_nan=False), encoding="utf-8")
    tmp.replace(path)

In [ ]:
# ---------------- SETTINGS ----------------
PROJECT_DIR = Path.cwd()          # same folder as the other notebooks
RUN_DIR_OVERRIDE = None           # optional: absolute path of a data snapshot
FIT_DIR_OVERRIDE = None           # optional: absolute path of a specific fit_* folder
BATCH_SIZE = 256                  # float32 inference; lower it if you hit CUDA out-of-memory
BLOCK = 5000                      # series per block, written to disk as it goes
LIMIT = None                      # e.g. 2000 for a quick smoke test
QUANTILES = [45, 55, 70, 75]
VERIFY_CACHE_HASHES = True        # re-hash every cached series file (about a minute); False to skip

# ---------------- DATA SNAPSHOT + FIT ----------------
RUN_DIR = Path(RUN_DIR_OVERRIDE or read_json(PROJECT_DIR / "iteration3_active_run.json")["run_dir"])
if not (RUN_DIR / "DATA_READY").exists():
    raise RuntimeError("Data snapshot is incomplete")
cfg = read_json(RUN_DIR / "config.json")
dm = read_json(RUN_DIR / "data_manifest.json")
for name, expected in [("config.json", dm["config_hash"]), ("calendar.parquet", dm["calendar_hash"]),
                       ("selected_series.parquet", dm["membership_hash"])]:
    if digest(RUN_DIR / name) != expected:
        raise RuntimeError(f"Data snapshot changed: {name}; create a new run")

TIME, KEY, TARGET = cfg["time_col"], cfg["group_col"], cfg["target_col"]
FUTURE = cfg["future_covariates"]
ICL, OCL = cfg["icl"], cfg["ocl"]
HISTORY_END = pd.Timestamp(cfg["val_end"])
FC_START, FC_END = pd.Timestamp(cfg["forecast_start"]), pd.Timestamp(cfg["forecast_end"])
HORIZON = (FC_END - FC_START).days + 1
assert HISTORY_END + pd.Timedelta(days=1) == FC_START
assert HORIZON == OCL, "Forecast window must equal output_chunk_length (no auto-regressive rollout)"

active = read_json(RUN_DIR / "active_fit.json")
FIT_DIR = Path(FIT_DIR_OVERRIDE or active["fit_dir"])
if FIT_DIR_OVERRIDE is None and digest(FIT_DIR / "bundle.json") != active["bundle_hash"]:
    raise RuntimeError("Bundle metadata changed")
bundle = read_json(FIT_DIR / "bundle.json")
if (bundle["run_id"] != cfg["run_id"] or bundle["config_hash"] != dm["config_hash"]
        or bundle["calendar_hash"] != dm["calendar_hash"]):
    raise RuntimeError("Checkpoint and data configuration do not match")
if darts.__version__ != bundle["versions"]["darts"]:
    raise RuntimeError("Use the same Darts version as training: " + bundle["versions"]["darts"])
for name, expected in bundle["file_hashes"].items():
    if digest(FIT_DIR / name) != expected:
        raise RuntimeError(f"Artifact changed: {name}")
MODEL_NAME = bundle["model_name"]
CACHE_DIR = FIT_DIR / "series_cache"
print("Fit folder :", FIT_DIR)
print(f"Forecast   : {FC_START.date()} -> {FC_END.date()} ({HORIZON} days) | history ends {HISTORY_END.date()}")

In [ ]:
# ---------------- SERIES, STATICS, COVARIATES ----------------
records = read_json(FIT_DIR / "cache_manifest.json")
with open(FIT_DIR / "static_encoded.pkl", "rb") as f:
    encoded = pickle.load(f)
static_raw = pd.read_parquet(FIT_DIR / "static_raw.parquet")
if len(encoded) != len(records) or static_raw[KEY].tolist() != [r["key"] for r in records]:
    raise RuntimeError("Series/static order mismatch")
if LIMIT:
    records, encoded = records[:LIMIT], encoded[:LIMIT]
keys = [r["key"] for r in records]
if not keys or len(set(keys)) != len(keys):
    raise ValueError("Forecast series keys must be non-empty and unique.")
if VERIFY_CACHE_HASHES:
    for r in records:
        if digest(CACHE_DIR / r["file"]) != r["sha256"]:
            raise RuntimeError("Cached target changed: " + r["key"])
print(f"Series to forecast: {len(keys):,}")

SHARED_COV = TimeSeries.from_pickle(FIT_DIR / "shared_cov.pkl")
if list(SHARED_COV.components) != FUTURE or SHARED_COV.end_time() < FC_END:
    raise RuntimeError("Future covariate schema or horizon mismatch")

# ---------------- MODEL (BEST checkpoint) ----------------
# Loading with best=True gives the epoch with the lowest val_loss, not the last epoch trained.
model = TFTModel.load_from_checkpoint(model_name=MODEL_NAME, work_dir=bundle["work_dir"],
                                      best=True, map_location="cpu")
best_files = list(Path(bundle["work_dir"]).rglob("best-*.ckpt"))
if best_files:
    print("Best checkpoint file:", max(best_files, key=lambda p: p.stat().st_ctime).name)
if model.input_chunk_length != ICL or model.output_chunk_length != OCL:
    raise RuntimeError("Checkpoint horizon mismatch")
if not isinstance(model.likelihood, NegativeBinomialLikelihood):
    raise TypeError("This notebook requires NegativeBinomialLikelihood.")

if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU not visible. Select your CUDA-enabled kernel; this will not fall back to CPU.")
torch.set_float32_matmul_precision("high")
# Own trainer: no loggers/checkpointing (the saved model would otherwise re-use its training logger),
# and float32 so the distribution parameters decode cleanly.
trainer = pl.Trainer(accelerator="gpu", devices=1, precision="32-true", logger=False,
                     enable_checkpointing=False, enable_progress_bar=False)
print("GPU:", torch.cuda.get_device_name(0), "| Darts:", darts.__version__, "| PyTorch:", torch.__version__)

In [ ]:
# ---------------- SEQUENCES ----------------
class History(Sequence):
    """Last ICL days of each series, ending the day before FC_START, read from the fit's cache."""

    def __init__(self, records, statics):
        self.records, self.statics = records, statics

    def __len__(self):
        return len(self.records)

    def __getitem__(self, i):
        if isinstance(i, slice):
            return [self[j] for j in range(*i.indices(len(self)))]
        if i < 0:
            i += len(self)
        if not 0 <= i < len(self):
            raise IndexError(i)
        with np.load(CACHE_DIR / self.records[i]["file"], allow_pickle=False) as z:
            dates = pd.DatetimeIndex(z["dates"])
            sales = z["sales"]
        m = (dates >= FC_START - pd.Timedelta(days=ICL)) & (dates <= HISTORY_END)
        ts = TimeSeries.from_times_and_values(dates[m], sales[m, None], columns=[TARGET],
                                              static_covariates=self.statics[i])
        if len(ts) != ICL or ts.end_time() != FC_START - pd.Timedelta(days=1):
            raise ValueError(f"History for {self.records[i]['key']} is not aligned to the forecast start.")
        return ts


class SharedCov(Sequence):
    def __init__(self, cov, n):
        self.cov, self.n = cov, n

    def __len__(self):
        return self.n

    def __getitem__(self, i):
        if isinstance(i, slice):
            return [self.cov for _ in range(*i.indices(self.n))]
        if i < 0:
            i += self.n
        if not 0 <= i < self.n:
            raise IndexError(i)
        return self.cov

In [ ]:
# ---------------- NEGATIVE-BINOMIAL DECODING ----------------
# Darts 0.40 returns (r, p) for torch.distributions.NegativeBinomial: mean = r*p/(1-p).
# SciPy's nbinom needs the probability 1-p. Darts' internal name "mu" is NOT the mean.

def summarise(p_ts):
    expected = [f"{TARGET}_r", f"{TARGET}_p"]
    if list(p_ts.components) != expected:
        raise ValueError(f"Expected {expected}, received {list(p_ts.components)}")
    v = p_ts.values(copy=False).astype(np.float64)
    r, pr = v[:, 0], v[:, 1]
    if not np.isfinite(v).all() or np.any(r <= 0) or np.any((pr <= 0) | (pr >= 1)):
        raise ValueError("Invalid negative-binomial parameters; inspect precision/output.")
    out = {"PRED_MEAN": r * pr / (1.0 - pr)}
    for q in QUANTILES:
        out[f"PRED_Q{q}"] = stats.nbinom.ppf(q / 100.0, r, 1.0 - pr)
    if not all(np.isfinite(x).all() for x in out.values()):
        raise ValueError("Non-finite decoded predictions.")
    return out


# Deterministic self-test of the installed likelihood (does not run the TFT network).
_raw = torch.tensor([[[[-2.0, -1.0]], [[0.4, -0.2]], [[3.0, 1.5]]]], dtype=torch.float64)
_lk = model.likelihood
_dist = _lk._distr_from_params(_lk._params_from_output(_raw))
_exported = _lk.predict_likelihood_parameters(_raw).detach().cpu().numpy().reshape(3, 2)
_probe = TimeSeries.from_times_and_values(pd.date_range("2000-01-01", periods=3), _exported,
                                          columns=[f"{TARGET}_r", f"{TARGET}_p"])
np.testing.assert_allclose(summarise(_probe)["PRED_MEAN"], _dist.mean.detach().cpu().numpy().ravel(),
                           rtol=1e-10, atol=1e-10)
_counts = torch.tensor([0.0, 2.0, 5.0], dtype=torch.float64).reshape(1, 3, 1)
np.testing.assert_allclose(stats.nbinom.logpmf(_counts.numpy().ravel(), _exported[:, 0], 1.0 - _exported[:, 1]),
                           _dist.log_prob(_counts).detach().cpu().numpy().ravel(), rtol=1e-10, atol=1e-10)
print("PASS: decoded means and SciPy probabilities match the installed likelihood.")

In [ ]:
# ---------------- FORECAST ----------------
OUT_DIR = FIT_DIR / ("predictions_" + datetime.now().strftime("%Y%m%d_%H%M%S_") + uuid4().hex[:8])
OUT_DIR.mkdir(exist_ok=False)
print("Output directory:", OUT_DIR)

n_blocks = (len(keys) + BLOCK - 1) // BLOCK
paths, t0 = [], datetime.now()
for b in range(n_blocks):
    lo, hi = b * BLOCK, min((b + 1) * BLOCK, len(keys))
    path = OUT_DIR / f"{MODEL_NAME}_block_{b:04d}.parquet"
    paths.append(path)

    preds = model.predict(
        n=HORIZON,
        series=History(records[lo:hi], encoded[lo:hi]),
        future_covariates=SharedCov(SHARED_COV, hi - lo),
        predict_likelihood_parameters=True,   # one pass instead of num_samples passes
        num_samples=1,
        batch_size=BATCH_SIZE,
        trainer=trainer,
        dataloader_kwargs={"num_workers": 0, "pin_memory": True},
        verbose=False,
    )
    if len(preds) != hi - lo:
        raise ValueError("Prediction count does not match the requested series.")
    for p_ts in preds:
        if len(p_ts) != HORIZON or p_ts.start_time() != FC_START or p_ts.end_time() != FC_END:
            raise ValueError("Forecast dates do not match the requested horizon.")

    pd.concat(
        [pd.DataFrame({KEY: k, TIME: p_ts.time_index, **summarise(p_ts)})
         for k, p_ts in zip(keys[lo:hi], preds)],
        ignore_index=True,
    ).to_parquet(path, index=False)

    el = (datetime.now() - t0).total_seconds()
    print(f"[{b+1}/{n_blocks}] {hi:,}/{len(keys):,} | {hi/el:,.1f} series/s | "
          f"ETA {(len(keys)-hi)/(hi/el)/60:.0f} min")
    del preds
    gc.collect()

if trainer.strategy.root_device.type != "cuda":
    raise RuntimeError("Prediction did not use CUDA")

# ---------------- OUTPUT ----------------
df = pd.concat([pd.read_parquet(p) for p in paths], ignore_index=True)
if len(df) != len(keys) * HORIZON or df.duplicated([KEY, TIME]).any():
    raise ValueError("Unexpected row count or duplicate series/date predictions.")
out = OUT_DIR / f"{MODEL_NAME}_predictions.parquet"
out_str = os.path.abspath(out)
if os.name == "nt":                       # long-path prefix for deep Windows folders
    out_str = "\\\\?\\" + out_str
df.to_parquet(out_str, index=False)
write_json(OUT_DIR / "prediction_manifest.json", dict(
    model_name=MODEL_NAME, run_id=cfg["run_id"], forecast_start=str(FC_START.date()),
    forecast_end=str(FC_END.date()), series_count=len(keys), rows=len(df), quantiles=QUANTILES,
    batch_size=BATCH_SIZE, precision="32-true", total_mean=float(df["PRED_MEAN"].sum())))

print(f"\nWritten -> {out}")
print(f"  rows: {len(df):,} | elapsed: {(datetime.now()-t0).total_seconds()/60:.1f} min")
for c in ["PRED_MEAN"] + [f"PRED_Q{q}" for q in QUANTILES]:
    print(f"  {c}: {df[c].sum():,.0f} ({df[c].sum()/1e5:.2f} lacs)")
print("Quantiles are per series-day; summing them does not give quantiles of total demand.")